In [ ]:
import pandas as pd
from ortools.sat.python import cp_model


pers_df = pd.read_excel('assignment_results_2.xlsx', sheet_name='Sheet1')
pos_df = pd.read_excel('Phase2.xlsx', sheet_name='Pos')

In [45]:
# Mapping
citizenship_map = {"Local": 1, "PR": 2, "Foreigner": 3, "Any": 4}
field_map = {"IT":1, "MED":2, "SALES":3, "ENGINEER":4, "RANDOM":5, "LEAD":6, "ANY":7}
tag_map = {"HR":1, "Int":2, "Ops":3, "Logs":4, "Plans":5, "Train":6, "Main":7}
tier_map = {"Junior":1, "Mid":2, "Senior":3}

pers_df["Pers_Cit"] = pers_df["Pers_Cit"].map(citizenship_map)
pers_df["Pers_Tier"] = pers_df["Pers_Tier"].map(tier_map)
pers_df["Pers_Field"] = pers_df["Pers_Field"].map(field_map)

pos_df["Req_Citizenship"] = pos_df["Req_Citizenship"].map(citizenship_map)
pos_df["Req_Tier"] = pos_df["Req_Tier"].map(tier_map)
pos_df["Req_Field"] = pos_df["Req_Field"].map(field_map)
pos_df["Tag"] = pos_df["Tag"].map(tag_map)


In [46]:
pers_df

,Person_ID,Person_Name,Pers_Cit,Pers_Tier,Pers_Field,Pass_1,Pass_2,Desk_ID,Req_Cit,Req_Tier,Req_Field
0,P000001,Roxuqy Tiwoni,1,2,1,True,False,D001995,Local,Mid,ANY
1,P000002,Lyxu Subyzu,3,1,6,False,False,D001976,Foreigner,Junior,LEAD
2,P000003,Cagyla Gedyzo,1,2,2,True,False,D001887,Local,Mid,MED
3,P000004,Huwyle Wola,2,2,6,True,False,D002066,PR,Mid,LEAD
4,P000005,Levyjo Buti,1,2,1,True,False,D001967,Local,Mid,IT
...,...,...,...,...,...,...,...,...,...,...,...
1992,P001993,Mozuni Ciqe,1,1,4,False,False,D000010,Local,Junior,ANY
1993,P001994,Vogu Sysu,1,2,1,True,False,D000034,Local,Mid,IT
1994,P001995,Tybufe Cotysy,1,1,4,False,False,D000008,Local,Junior,ENGINEER
1995,P001996,Viji Zuqemi,1,2,1,True,False,D000003,Local,Mid,ANY


In [47]:
model = cp_model.CpModel()
solver = cp_model.CpSolver()

no_of_people = len(pers_df)
no_of_desk = len(pos_df)

assignments = {}
penalties   = {}

In [ ]:
for i in range(no_of_people):
    pers = pers_df.loc[i]
    
    # * avoids assigning a desk that personnel has taken in the past *
    previous_desks = pers["Past_Desks"].split(",")
    previous_desks.append(pers["Desk_ID"])
    
    # previous_desks = {pers["Desk_ID"]}
    for j in range(no_of_desk):
        pos  = pos_df.loc[j]
        
        if pos["Desk_ID"] in previous_desks:
            assignments[i,j] = model.NewBoolVar(f"x_{i}_{j}")
            model.Add(assignments[i, j] == 0)
            continue
        
        
        
        
        citizen_match = (pers["Pers_Cit"] == pos["Req_Citizenship"]) or (pos["Req_Citizenship"] == 4)
        field_match   = (pers["Pers_Field"] == pos["Req_Field"]) or (pos["Req_Field"] == 7)

        if citizen_match and field_match:
            assignments[i,j] = model.NewBoolVar(f"x_{i}_{j}")

            penalty_var = model.NewIntVar(0, 10, f"penalty_{i}_{j}")
            penalties[i,j] = penalty_var

            pers_tier = pers["Pers_Tier"]
            pos_tier  = pos["Req_Tier"]

            if pers_tier == pos_tier:
                model.Add(penalty_var == 0).OnlyEnforceIf(assignments[i,j])

            elif (pers_tier == 1 and pos_tier == 2) or \
                 (pers_tier == 2 and pos_tier == 1) or \
                 (pers_tier == 2 and pos_tier == 3) or \
                 (pers_tier == 3 and pos_tier == 2):
                model.Add(penalty_var == 1).OnlyEnforceIf(assignments[i,j])



In [49]:
for i in range(no_of_people):
    feasible_jobs = [j for j in range(no_of_desk) if (i,j) in assignments]
    if feasible_jobs:
        model.Add(sum(assignments[i,j] for j in feasible_jobs) == 1)
    else:
        print(f"Warning: Person {pers_df.loc[i,'Name_ID']} has no feasible job!")
        
for j in range(no_of_desk): 
    feasible_people = [i for i in range(no_of_people) if (i,j) in assignments]
    if feasible_people:
        model.Add(sum(assignments[i,j] for i in feasible_people) <= 1)


In [50]:
appointment_mismatch = model.NewIntVar(0, no_of_people * no_of_desk * 10, "appointment_mismatch")
penalty_sum_list = [penalties[i,j] for i,j in penalties]
model.Add(appointment_mismatch == sum(penalty_sum_list))

model.Maximize(sum(assignments.values()) - appointment_mismatch)

# First solve
status_code = solver.Solve(model)
print(f"{solver.StatusName(status_code)} ({status_code})")
print("Objective value:", solver.ObjectiveValue())
print("Best bound:", solver.BestObjectiveBound())

# Solve again
relaxed_limit = int(solver.ObjectiveValue() * 1.05)

relaxed_model = cp_model.CpModel()
assignments_relaxed = {}
penalties_relaxed   = {}

for (i,j), var in assignments.items():
    assignments_relaxed[i,j] = relaxed_model.NewBoolVar(f"x_{i}_{j}")
    penalties_relaxed[i,j]   = relaxed_model.NewIntVar(0, 10, f"penalty_{i}_{j}")

appointment_mismatch_relaxed = relaxed_model.NewIntVar(0, no_of_people * no_of_desk * 10, "appointment_mismatch_relaxed")
relaxed_model.Add(appointment_mismatch_relaxed == sum(penalties_relaxed.values()))
relaxed_model.Add(appointment_mismatch_relaxed <= relaxed_limit)

relaxed_model.Maximize(sum(assignments_relaxed.values()) - appointment_mismatch_relaxed)

relaxed_solver = cp_model.CpSolver()
status_code = relaxed_solver.Solve(relaxed_model)
print(f"Relaxed solve: {relaxed_solver.StatusName(status_code)} ({status_code})")
print("Objective value (relaxed):", relaxed_solver.ObjectiveValue())


OPTIMAL (4)
Objective value: 1997.0
Best bound: 1997.0
Relaxed solve: OPTIMAL (4)
Objective value (relaxed): 1133746.0


In [51]:
pers_df

,Person_ID,Person_Name,Pers_Cit,Pers_Tier,Pers_Field,Pass_1,Pass_2,Desk_ID,Req_Cit,Req_Tier,Req_Field
0,P000001,Roxuqy Tiwoni,1,2,1,True,False,D001995,Local,Mid,ANY
1,P000002,Lyxu Subyzu,3,1,6,False,False,D001976,Foreigner,Junior,LEAD
2,P000003,Cagyla Gedyzo,1,2,2,True,False,D001887,Local,Mid,MED
3,P000004,Huwyle Wola,2,2,6,True,False,D002066,PR,Mid,LEAD
4,P000005,Levyjo Buti,1,2,1,True,False,D001967,Local,Mid,IT
...,...,...,...,...,...,...,...,...,...,...,...
1992,P001993,Mozuni Ciqe,1,1,4,False,False,D000010,Local,Junior,ANY
1993,P001994,Vogu Sysu,1,2,1,True,False,D000034,Local,Mid,IT
1994,P001995,Tybufe Cotysy,1,1,4,False,False,D000008,Local,Junior,ENGINEER
1995,P001996,Viji Zuqemi,1,2,1,True,False,D000003,Local,Mid,ANY


In [52]:
inv_citizenship_map = {v: k for k, v in citizenship_map.items()}
inv_field_map = {v: k for k, v in field_map.items()}
inv_tag_map = {v: k for k, v in tag_map.items()}
inv_tier_map = {v: k for k, v in tier_map.items()}

pers_df["Pers_Cit"] = pers_df["Pers_Cit"].map(inv_citizenship_map)
pers_df["Pers_Tier"] = pers_df["Pers_Tier"].map(inv_tier_map)
pers_df["Pers_Field"] = pers_df["Pers_Field"].map(inv_field_map)

pos_df["Req_Citizenship"] = pos_df["Req_Citizenship"].map(inv_citizenship_map)
pos_df["Req_Tier"] = pos_df["Req_Tier"].map(inv_tier_map)
pos_df["Req_Field"] = pos_df["Req_Field"].map(inv_field_map)
pos_df["Tag"] = pos_df["Tag"].map(inv_tag_map)

In [53]:
pers_df

,Person_ID,Person_Name,Pers_Cit,Pers_Tier,Pers_Field,Pass_1,Pass_2,Desk_ID,Req_Cit,Req_Tier,Req_Field
0,P000001,Roxuqy Tiwoni,Local,Mid,IT,True,False,D001995,Local,Mid,ANY
1,P000002,Lyxu Subyzu,Foreigner,Junior,LEAD,False,False,D001976,Foreigner,Junior,LEAD
2,P000003,Cagyla Gedyzo,Local,Mid,MED,True,False,D001887,Local,Mid,MED
3,P000004,Huwyle Wola,PR,Mid,LEAD,True,False,D002066,PR,Mid,LEAD
4,P000005,Levyjo Buti,Local,Mid,IT,True,False,D001967,Local,Mid,IT
...,...,...,...,...,...,...,...,...,...,...,...
1992,P001993,Mozuni Ciqe,Local,Junior,ENGINEER,False,False,D000010,Local,Junior,ANY
1993,P001994,Vogu Sysu,Local,Mid,IT,True,False,D000034,Local,Mid,IT
1994,P001995,Tybufe Cotysy,Local,Junior,ENGINEER,False,False,D000008,Local,Junior,ENGINEER
1995,P001996,Viji Zuqemi,Local,Mid,IT,True,False,D000003,Local,Mid,ANY


In [55]:
assigned_rows = []
for (i,j), var in assignments.items():
    if solver.Value(var):
        assigned_rows.append({
            "Person_ID": pers_df.loc[i, "Person_ID"],
            "Person_Name": pers_df.loc[i, "Person_Name"],
            "Pers_Cit": pers_df.loc[i, "Pers_Cit"],
            "Pers_Tier": pers_df.loc[i, "Pers_Tier"],
            "Pers_Field": pers_df.loc[i, "Pers_Field"],
            "Pass_1": pers_df.loc[i, "Pass_1"],
            "Pass_2": pers_df.loc[i, "Pass_2"],
            "Desk_ID": pos_df.loc[j, "Desk_ID"],
            "Req_Cit": pos_df.loc[j, "Req_Citizenship"],
            "Req_Tier": pos_df.loc[j, "Req_Tier"],
            "Req_Field": pos_df.loc[j, "Req_Field"],
            "Past_Desk": pers_df.loc[i, "Desk_ID"],
        })

assigned_df = pd.DataFrame(assigned_rows)
assigned_df.to_excel("assignment_results_2.xlsx", index=False)
print("Assignments saved to assignment_results_2.xlsx")


Assignments saved to assignment_results_2.xlsx
